In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

## 加载LLM

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

## 定义tools

In [3]:
from langchain.agents import tool

### generate_unique_timestamp

In [4]:
import time

@tool
def generate_unique_timestamp():
    """
    生成唯一的时间戳。输入始终为空字符串。

    Returns:
        int: 唯一的时间戳，以毫秒为单位。
    """
    timestamp = int(time.time() * 1000)  # 获取当前时间的毫秒级时间戳
    return timestamp

### create_folder

In [5]:
import os

@tool
def create_folder(folder_name):
    """
    根据给定的文件夹名创建文件夹。

    Args:
        folder_name (str): 要创建的文件夹的名称。

    Returns:
        str: 创建的文件夹的路径。
    """
    try:
        os.makedirs(os.path.join("chat_history", folder_name))  # 创建文件夹
        return os.path.abspath(folder_name)  # 返回创建的文件夹的绝对路径
    except OSError as e:
        print(f"创建文件夹失败：{e}")
        return None

### delete_temp_folder

In [6]:
import shutil

@tool
def delete_temp_folder():
    """
    删除 chat_history 文件夹下的 temp 文件夹。输入始终为空字符串。

    Returns:
        bool: 如果成功删除则返回 True，否则返回 False。
    """
    temp_folder = "chat_history/temp"  # temp 文件夹路径

    try:
        shutil.rmtree(temp_folder)  # 递归删除 temp 文件夹及其所有内容
        print("成功删除 temp 文件夹。")
        return True
    except Exception as e:
        print(f"删除 temp 文件夹失败：{e}")
        return False

### copy_chat_history

In [7]:
@tool
def copy_chat_history(interview_id: str) -> str:
    """
    将 chat_history/temp 文件夹中的 chat_history.txt 文件复制到 chat_history 文件夹下的以 interview_id 命名的子文件夹中。
    如果面试ID文件夹不存在，则返回相应的提示字符串。

    参数:
        interview_id (str): 面试的唯一标识符。

    返回:
        str: 操作结果的提示信息。
    """
    # 确定临时文件夹和面试文件夹路径
    temp_folder = os.path.join("chat_history", "temp")
    interview_folder = os.path.join("chat_history", interview_id)

    # 检查面试文件夹是否存在
    if not os.path.exists(interview_folder):
        return f"面试ID为 {interview_id} 的文件夹不存在。无法完成复制操作。"

    # 确定源文件和目标文件的路径
    source_file = os.path.join(temp_folder, 'chat_history.txt')
    destination_file = os.path.join(interview_folder, 'chat_history.txt')

    # 检查源文件是否存在
    if not os.path.exists(source_file):
        return f"源文件 {source_file} 不存在。无法完成复制操作。"

    # 将 chat_history.txt 从临时文件夹复制到面试文件夹
    shutil.copyfile(source_file, destination_file)

    return f"已将 chat_history.txt 复制到面试ID为 {interview_id} 的文件夹中。"

### read_chat_history

In [8]:
@tool
def read_chat_history(interview_id: str) -> str:
    """
    读取指定面试ID文件夹下的聊天记录(chat_history.txt)内容。

    参数:
        interview_id (str): 面试的唯一标识符。

    返回:
        str: 聊天记录的内容。
    """
    # 确定面试文件夹路径
    interview_folder = os.path.join("chat_history", interview_id)

    # 检查面试文件夹是否存在
    if not os.path.exists(interview_folder):
        return f"面试ID为 {interview_id} 的文件夹不存在。无法读取聊天记录。"

    # 确定聊天记录文件路径
    chat_history_file = os.path.join(interview_folder, 'chat_history.txt')

    # 检查聊天记录文件是否存在
    if not os.path.exists(chat_history_file):
        return f"聊天记录文件 {chat_history_file} 不存在。无法读取内容。"

    # 读取聊天记录文件内容
    with open(chat_history_file, 'r', encoding='utf-8') as file:
        chat_history_content = file.read()

    return chat_history_content

### generate_markdown_file

In [9]:
@tool
def generate_markdown_file(interview_id: str, interview_feedback: str) -> str:
    """
    将给定的面试反馈内容生成为 Markdown 文件，并保存到指定的面试ID文件夹中。

    参数:
        interview_id (str): 面试的唯一标识符。
        interview_feedback (str): 面试反馈的内容。

    返回:
        str: 操作结果的提示信息。
    """
    # 确定面试文件夹路径
    interview_folder = os.path.join("chat_history", interview_id)

    # 检查面试文件夹是否存在
    if not os.path.exists(interview_folder):
        return f"面试ID为 {interview_id} 的文件夹不存在。无法生成 Markdown 文件。"

    # 生成 Markdown 文件路径
    markdown_file_path = os.path.join(interview_folder, "面试报告.md")

    try:
        # 写入 Markdown 文件
        with open(markdown_file_path, 'w', encoding='utf-8') as file:
            # 写入标题和面试反馈
            file.write("# 面试报告\n\n")
            file.write("## 面试反馈：\n\n")
            file.write(interview_feedback)
            file.write("\n\n")

            # 读取 chat_history.txt 文件内容并写入 Markdown 文件
            chat_history_file_path = os.path.join(interview_folder, "chat_history.txt")
            if os.path.exists(chat_history_file_path):
                file.write("## 面试记录：\n\n")
                with open(chat_history_file_path, 'r', encoding='utf-8') as chat_file:
                    for line in chat_file:
                        file.write(line.rstrip('\n') + '\n\n')  # 添加换行符

        return f"已生成 Markdown 文件: {markdown_file_path}"
    except Exception as e:
        return f"生成 Markdown 文件时出错: {str(e)}"

### find_most_relevant_block_from_cv

In [10]:
from utils import parse_cv_to_md
cv_file_path = "data/cv.txt"
result = parse_cv_to_md(llm, cv_file_path)
print(result)


基于简历文本，按照约束，转换成Markdown格式：

简历文本：
[{cv_content}]

约束：
1、只用一级标题和二级标题分出来简历的大块和小块
2、一级标题只有这些：个人信息、教育经历、工作经历、项目经历、校园经历、职业技能、曾获奖项、兴趣爱好、自我评价、其他信息。

Markdown：

```markdown
# 个人信息
姓名：张三  
年龄：28  
性别：男  
联系方式：12312341234 | zhangsan@example.com  
目前具体薪酬：保密  
应聘岗位：大模型应用开发工程师/NLP工程师  
期望地点：北京  
在职状态：在职，寻求更好的发展机会  
换工作原因：寻求新的技术挑战和更广阔的职业发展空间  

# 教育经历
xx大学 | 计算机科学与技术（硕士） | 2016年9月 - 2018年7月  
xx大学 | 计算机科学与技术（本科） | 2012年9月 - 2016年7月  

# 工作经历
## xx公司 | NLP工程师 | 2018年8月 - 至今
职责和成就：
- 负责智能问答系统的设计和开发，通过引入langchain和RAG技术，显著提升了问题解答的准确率和效率。
- 实施Agent驱动的自动化内容管理系统，大幅提高内容处理能力和用户交互体验。
- 引领团队采用提示工程优化模型性能，成功将项目落地应用，为公司带来显著的经济效益。

## xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月
职责和成就：
- 负责设计和实现公司的NLP核心算法，包括命名实体识别、情感分析等，以提升自然语言处理系统的性能和准确性。
- 利用Pandas和Numpy进行数据预处理和清洗，有效处理了超过10TB的文本数据，提高了数据处理效率和模型训练速度。
- 使用Sklearn实现机器学习模型，通过优化模型参数和特征工程显著提升了命名实体识别和情感分析的准确率。
- 开发自动化脚本监控模型性能，定期调整算法以适应新的数据集，确保系统的持续优化和更新。

# 项目经历
## 基于LangChain的智能问答系统（在线教育平台）
- **任务**：为一家在线教育平台设计并实现一个基于LangChain的智能问答系统，旨在提供即时、准确的学术支持和课程咨询服务。
- **难点**：整合广泛的教育资源

In [17]:
from utils import parse_md_file_to_docs

file_path = "data/cv.md"
docs = parse_md_file_to_docs(file_path)
print(len(docs))
docs

11


[Document(page_content='# None\n## None\nmarkdown'),
 Document(metadata={'Title 1': '个人信息'}, page_content='# 个人信息\n## None\n姓名：张三\n年龄：28\n性别：男\n联系方式：12312341234 | zhangsan@example.com\n目前具体薪酬：保密\n应聘岗位：大模型应用开发工程师/NLP工程师\n期望地点：北京\n在职状态：在职，寻求更好的发展机会\n换工作原因：寻求新的技术挑战和更广阔的职业发展空间'),
 Document(metadata={'Title 1': '教育经历'}, page_content='# 教育经历\n## None\nxx大学 | 计算机科学与技术（硕士） | 2016年9月 - 2018年7月\nxx大学 | 计算机科学与技术（本科） | 2012年9月 - 2016年7月'),
 Document(metadata={'Title 1': '工作经历', 'Title 2': 'xx公司 | NLP工程师 | 2018年8月 - 至今'}, page_content='# 工作经历\n## xx公司 | NLP工程师 | 2018年8月 - 至今\n职责和成就：\n- 负责智能问答系统的设计和开发，通过引入langchain和RAG技术，显著提升了问题解答的准确率和效率。\n- 实施Agent驱动的自动化内容管理系统，大幅提高内容处理能力和用户交互体验。\n- 引领团队采用提示工程优化模型性能，成功将项目落地应用，为公司带来显著的经济效益。'),
 Document(metadata={'Title 1': '工作经历', 'Title 2': 'xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月'}, page_content='# 工作经历\n## xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月\n职责和成就：\n- 负责设计和实现公司的NLP核心算法，包括命名实体识别、情感分析等，以提升自然语言处理系统的性能和准确性。\n- 利用Pandas和Numpy进行数据预处理和清洗，有效处理了超过10TB的文本数据，提高了数据处理效率和模型训练

In [18]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [19]:
retriever.invoke("langchain")

[Document(metadata={'Title 1': '项目经历', 'Title 2': '基于LangChain的智能问答系统（在线教育平台）'}, page_content='# 项目经历\n## 基于LangChain的智能问答系统（在线教育平台）\n- **任务**：为一家在线教育平台设计并实现一个基于LangChain的智能问答系统，旨在提供即时、准确的学术支持和课程咨询服务。\n- **难点**：整合广泛的教育资源和文献，提高问答系统对学术术语的理解能力和回答质量。\n- **解决方案**：与教育专家合作开发定制化的知识图谱，利用LangChain进行高效的信息检索和生成，确保回答的专业性和准确性。\n- **成果**：系统上线3个月内，用户查询响应时间缩短50%，平台用户满意度和留存率显著提升。'),
 Document(metadata={'Title 1': '项目经历', 'Title 2': '基于LangChain的智能问答系统（在线教育平台）'}, page_content='# 项目经历\n## 基于LangChain的智能问答系统（在线教育平台）\n- **任务**：为一家在线教育平台设计并实现一个基于LangChain的智能问答系统，旨在提供即时、准确的学术支持和课程咨询服务。\n- **难点**：整合广泛的教育资源和文献，提高问答系统对学术术语的理解能力和回答质量。\n- **解决方案**：与教育专家合作开发定制化的知识图谱，利用LangChain进行高效的信息检索和生成，确保回答的专业性和准确性。\n- **成果**：系统上线3个月内，用户查询响应时间缩短50%，平台用户满意度和留存率显著提升。'),
 Document(metadata={'Title 1': '工作经历', 'Title 2': 'xx公司 | NLP工程师 | 2018年8月 - 至今'}, page_content='# 工作经历\n## xx公司 | NLP工程师 | 2018年8月 - 至今\n职责和成就：\n- 负责智能问答系统的设计和开发，通过引入langchain和RAG技术，显著提升了问题解答的准确率和效率。\n- 实施Agent驱动的自动化内容管理系统，大幅提高内容处理能力和用户交互体验。\n- 引领团队采用提示工程优化模型性能，成

In [21]:
retriever.invoke("nlp")

[Document(metadata={'Title 1': '工作经历', 'Title 2': 'xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月'}, page_content='# 工作经历\n## xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月\n职责和成就：\n- 负责设计和实现公司的NLP核心算法，包括命名实体识别、情感分析等，以提升自然语言处理系统的性能和准确性。\n- 利用Pandas和Numpy进行数据预处理和清洗，有效处理了超过10TB的文本数据，提高了数据处理效率和模型训练速度。\n- 使用Sklearn实现机器学习模型，通过优化模型参数和特征工程显著提升了命名实体识别和情感分析的准确率。\n- 开发自动化脚本监控模型性能，定期调整算法以适应新的数据集，确保系统的持续优化和更新。'),
 Document(metadata={'Title 1': '工作经历', 'Title 2': 'xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月'}, page_content='# 工作经历\n## xx科技公司 | NLP算法工程师 | 2016年7月 - 2018年7月\n职责和成就：\n- 负责设计和实现公司的NLP核心算法，包括命名实体识别、情感分析等，以提升自然语言处理系统的性能和准确性。\n- 利用Pandas和Numpy进行数据预处理和清洗，有效处理了超过10TB的文本数据，提高了数据处理效率和模型训练速度。\n- 使用Sklearn实现机器学习模型，通过优化模型参数和特征工程显著提升了命名实体识别和情感分析的准确率。\n- 开发自动化脚本监控模型性能，定期调整算法以适应新的数据集，确保系统的持续优化和更新。'),
 Document(metadata={'Title 1': '自我评价'}, page_content='# 自我评价\n## None\n自我反思：具有深厚的计算机科学理论基础和丰富的NLP实践经验，擅长通过技术创新解决复杂问题。\n专业技能：精通Python编程、PyTorch、LangChain、Agent架构；熟悉NLP相关算法和预训练模型，如Transformer、BERT、GPT等。熟练使用Pandas、Numpy、Sklearn等数

In [26]:
@tool
def find_most_relevant_block_from_cv(sentence: str) -> str:
    """
    当你需要根据职位描述（JD）中的技能关键词去简历文本中找到相关内容时，就可以调用这个函数。

    参数:
        sentence (str): 包含技能关键词的句子。

    返回:
        str: 最相关的文本块。
    """
    try:
        most_relevant_docs = retriever.invoke(sentence)
        print(len(most_relevant_docs))

        if most_relevant_docs:
            most_relevant_texts = [doc.page_content for doc in most_relevant_docs]
            most_relevant_text = "\n\n".join(most_relevant_texts)
            return most_relevant_text
        else:
            return "未找到相关文本块"
    except Exception as e:
        print(f"find_most_relevant_block_from_cv()发生错误：{e}")
        return "函数发生错误，未找到相关文本块"

In [27]:
print(find_most_relevant_block_from_cv.invoke("langchain"))

3
# 项目经历
## 基于LangChain的智能问答系统（在线教育平台）
- **任务**：为一家在线教育平台设计并实现一个基于LangChain的智能问答系统，旨在提供即时、准确的学术支持和课程咨询服务。
- **难点**：整合广泛的教育资源和文献，提高问答系统对学术术语的理解能力和回答质量。
- **解决方案**：与教育专家合作开发定制化的知识图谱，利用LangChain进行高效的信息检索和生成，确保回答的专业性和准确性。
- **成果**：系统上线3个月内，用户查询响应时间缩短50%，平台用户满意度和留存率显著提升。

# 项目经历
## 基于LangChain的智能问答系统（在线教育平台）
- **任务**：为一家在线教育平台设计并实现一个基于LangChain的智能问答系统，旨在提供即时、准确的学术支持和课程咨询服务。
- **难点**：整合广泛的教育资源和文献，提高问答系统对学术术语的理解能力和回答质量。
- **解决方案**：与教育专家合作开发定制化的知识图谱，利用LangChain进行高效的信息检索和生成，确保回答的专业性和准确性。
- **成果**：系统上线3个月内，用户查询响应时间缩短50%，平台用户满意度和留存率显著提升。

# 工作经历
## xx公司 | NLP工程师 | 2018年8月 - 至今
职责和成就：
- 负责智能问答系统的设计和开发，通过引入langchain和RAG技术，显著提升了问题解答的准确率和效率。
- 实施Agent驱动的自动化内容管理系统，大幅提高内容处理能力和用户交互体验。
- 引领团队采用提示工程优化模型性能，成功将项目落地应用，为公司带来显著的经济效益。


### 创建tools

In [28]:
tools = [
    generate_unique_timestamp, 
    create_folder, 
    delete_temp_folder, 
    copy_chat_history, 
    read_chat_history, 
    generate_markdown_file,
    find_most_relevant_block_from_cv
]

## 创建prompt

In [29]:
from utils import *

In [30]:
jd_file_path = "data/jd.txt"
jd_json_file_path = parse_jd_to_json(llm, jd_file_path)
jd_json_file_path


基于JD文本，按照约束，生成以下格式的 JSON 数据：
{{
  "基本信息": {{
    "职位": "职位名称",
    "薪资": "薪资范围",
    "地点": "工作地点",
    "经验要求": "经验要求",
    "学历要求": "学历要求",
    "其他":""
  }},
  "岗位职责": {{
    "具体职责": ["职责1", "职责2", ...]
  }},
  "岗位要求": {{
    "学历背景": "学历要求",
    "工作经验": "工作经验要求",
   "技能要求": ["技能1", "技能2", ...],
    "个人特质": ["特质1", "特质2", ...],
  }},
  "专业技能/知识/能力": ["技能1", "技能2", ...],
  "其他信息": {{}}
}}

JD文本：
[{jd_content}]

约束：
1、除了`专业技能/知识/能力`键，其他键的值都从原文中获取。
2、保证JSON里的值全面覆盖JD原文，不遗漏任何原文，不知如何分类就放到`其他信息`里。
3、`专业技能/知识/能力`键对应的值要求从JD全文中（尤其是岗位职责、技能要求部分）提取总结关键词或关键短句，不能有任何遗漏的硬技能。

JSON：

{'基本信息': {'职位': 'Python工程师 (AI应用方向)', '薪资': '11-20K·14薪', '地点': '北京', '经验要求': '1-3年', '学历要求': '本科', '其他': ''}, '岗位职责': {'具体职责': ['参与开发基于语音和数据的企业AI助理产品', '开发及维护基于大模型的中间层服务，处理语音、文本、文档、链接以及中台数据等形成知识库']}, '岗位要求': {'学历背景': '计算机科学或相关领域的学士以上学位，或有相关工作经验', '工作经验': '1-5年Python开发经验', '技能要求': ['熟悉AI开发工具集，如OpenAI、LangChain、向量数据库、Text-SQL等', '熟练至少一种Python Web框架如FastAPI和ORM框架', '熟练编写Prompt提示词，利用AI的能力快速构建应用'], '个人特质': ['有良好的沟通和团队合作能力', '有较强

'data/jd.json'

In [31]:
jd_json_file_path = "data/jd.json"
jd_dict = read_json(jd_json_file_path)
jd_dict

{'基本信息': {'职位': 'Python工程师 (AI应用方向)',
  '薪资': '11-20K·14薪',
  '地点': '北京',
  '经验要求': '1-3年',
  '学历要求': '本科',
  '其他': ''},
 '岗位职责': {'具体职责': ['参与开发基于语音和数据的企业AI助理产品',
   '开发及维护基于大模型的中间层服务，处理语音、文本、文档、链接以及中台数据等形成知识库']},
 '岗位要求': {'学历背景': '计算机科学或相关领域的学士以上学位，或有相关工作经验',
  '工作经验': '1-5年Python开发经验',
  '技能要求': ['熟悉AI开发工具集，如OpenAI、LangChain、向量数据库、Text-SQL等',
   '熟练至少一种Python Web框架如FastAPI和ORM框架',
   '熟练编写Prompt提示词，利用AI的能力快速构建应用'],
  '个人特质': ['有良好的沟通和团队合作能力', '有较强的自我驱动力和问题解决能力', '有强烈的好奇心，热爱学习新鲜事物']},
 '专业技能/知识/能力': ['Python',
  'PyTorch',
  'TensorFlow',
  'Numpy',
  'Redis',
  'MySQL',
  'openai',
  'langchain'],
 '其他信息': {}}

In [32]:
job_title = jd_dict.get('基本信息').get('职位')
job_key_skills = jd_dict.get('专业技能/知识/能力')
print(f"职位：{job_title}")
print(f"专业技能/知识/能力：{job_key_skills}")

职位：Python工程师 (AI应用方向)
专业技能/知识/能力：['Python', 'PyTorch', 'TensorFlow', 'Numpy', 'Redis', 'MySQL', 'openai', 'langchain']


In [33]:
system_prompt = f"""
## Role and Goals
- 你是所招岗位“{job_title}”的技术专家，同时也作为技术面试官向求职者提出技术问题，专注于考察应聘者的专业技能、知识和能力。
- 这里是当前岗位所需的专业技能、知识和能力：“{job_key_skills}”，你应该重点围绕这些技术点提出你的问题。
- 你严格遵守面试流程进行面试。

## Interview Workflow
1. 当应聘者说开始面试后，
    1.1 你要依据当前时间生成一个新的时间戳作为面试ID（只会在面试开始的时候生成面试ID，其他任何时间都不会）
    1.2 以该面试ID为文件夹名创建本地文件夹（只会在面试开始的时候创建以面试ID为名的文件夹，其他任何时间都不会）
    1.3 删除存储聊天记录的临时文件夹
    1.4 输出该面试ID给应聘者，并结合当前技术点、与技术点相关的简历内容，提出你的第一个基础技术问题。
2. 接收应聘者的回答后，
    2.1 检查应聘者的回答是否有效
        2.1.1 如果是对面试官问题的正常回答（无论回答的好不好，还是回答不会，都算正常回答），就跳转到2.2处理
        2.1.2 如果是与面试官问题无关的回答（胡言乱语、辱骂等），请警告求职者需要严肃对待面试，跳过2.2，再次向求职者提出上次的问题。
    2.2 如果应聘者对上一个问题回答的很好，就基于当前技术点和历史记录提出一个更深入一点的问题；
        如果应聘者对上一个问题回答的一般，就基于当前技术点和历史记录提出另一个角度的问题；
        如果应聘者对上一个问题回答的不好，就基于当前技术点和历史记录提出一个更简单一点的问题；
        如果应聘者对上一个问题表示不会、不懂、一点也回答不了，就换一个与当前技术点不同的技术点进行技术提问。
3. 当应聘者想结束面试或当应聘者想要面试报告，
    3.1 从临时文件夹里复制一份聊天记录文件到当前面试ID文件夹下。
    3.2 读取当前面试ID文件夹下的聊天记录，基于聊天记录、从多个角度评估应聘者的表现、生成一个详细的面试报告。
    3.3 调用工具生成一个面试报告的markdown文件到当前面试ID文件夹下
    3.4 告知应聘者面试已结束，以及面试报告的位置。
    
## Output Constraints
- 你发送给应聘者的信息中，一定不要解答你提出的面试问题，只需要有简短的反馈和提出的新问题。
- 你每次提出的技术问题，都需要结合从JD里提取的技术点和与技术点相关的简历内容，当你需要获取`与技术点相关的简历内容`时，请调用工具。
- 再一次检查你的输出，你一次只会问一个技术问题。
"""

In [34]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            system_prompt,
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [35]:
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]], 'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template="\n## Role and Goals\n- 你是所招岗位“Python工程师 (AI应用方向)”的技术专家，同时也作为技术面试官向求职者提出技术问题，专注于考察应聘者的专业技能、知识和能力。\n- 这里是当前岗位所需的专业技能、知识和能力：“['Python', 'PyTorch', 'TensorFlow', 'Numpy', 'Redis', 'MySQL', 'openai', 'lang

In [36]:
print(prompt.messages[0].prompt.template)


## Role and Goals
- 你是所招岗位“Python工程师 (AI应用方向)”的技术专家，同时也作为技术面试官向求职者提出技术问题，专注于考察应聘者的专业技能、知识和能力。
- 这里是当前岗位所需的专业技能、知识和能力：“['Python', 'PyTorch', 'TensorFlow', 'Numpy', 'Redis', 'MySQL', 'openai', 'langchain']”，你应该重点围绕这些技术点提出你的问题。
- 你严格遵守面试流程进行面试。

## Interview Workflow
1. 当应聘者说开始面试后，
    1.1 你要依据当前时间生成一个新的时间戳作为面试ID（只会在面试开始的时候生成面试ID，其他任何时间都不会）
    1.2 以该面试ID为文件夹名创建本地文件夹（只会在面试开始的时候创建以面试ID为名的文件夹，其他任何时间都不会）
    1.3 删除存储聊天记录的临时文件夹
    1.4 输出该面试ID给应聘者，并结合当前技术点、与技术点相关的简历内容，提出你的第一个基础技术问题。
2. 接收应聘者的回答后，
    2.1 检查应聘者的回答是否有效
        2.1.1 如果是对面试官问题的正常回答（无论回答的好不好，还是回答不会，都算正常回答），就跳转到2.2处理
        2.1.2 如果是与面试官问题无关的回答（胡言乱语、辱骂等），请警告求职者需要严肃对待面试，跳过2.2，再次向求职者提出上次的问题。
    2.2 如果应聘者对上一个问题回答的很好，就基于当前技术点和历史记录提出一个更深入一点的问题；
        如果应聘者对上一个问题回答的一般，就基于当前技术点和历史记录提出另一个角度的问题；
        如果应聘者对上一个问题回答的不好，就基于当前技术点和历史记录提出一个更简单一点的问题；
        如果应聘者对上一个问题表示不会、不懂、一点也回答不了，就换一个与当前技术点不同的技术点进行技术提问。
3. 当应聘者想结束面试或当应聘者想要面试报告，
    3.1 从临时文件夹里复制一份聊天记录文件到当前面试ID文件夹下。
    3.2 读取当前面试ID文件夹下的聊天记录，基于聊天记录、从多个角度评估应聘者的表现、生成一个详细的面试报告。
    3.3 调

## 将tools绑定到LLM

In [37]:
llm_with_tools = llm.bind_tools(tools)

In [38]:
llm_with_tools

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x000001826BE020E0>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x000001826BE03C10>, model_name='gpt-4o-mini', openai_api_key=SecretStr('**********'), openai_proxy=''), kwargs={'tools': [{'type': 'function', 'function': {'name': 'generate_unique_timestamp', 'description': '生成唯一的时间戳。输入始终为空字符串。\n\nReturns:\n    int: 唯一的时间戳，以毫秒为单位。', 'parameters': {'type': 'object', 'properties': {}}}}, {'type': 'function', 'function': {'name': 'create_folder', 'description': '根据给定的文件夹名创建文件夹。\n\nArgs:\n    folder_name (str): 要创建的文件夹的名称。\n\nReturns:\n    str: 创建的文件夹的路径。', 'parameters': {'type': 'object', 'properties': {'folder_name': {}}, 'required': ['folder_name']}}}, {'type': 'function', 'function': {'name': 'delete_temp_folder', 'description': '删除 chat_history 文件夹下的 temp 文件夹。输入始终为空字符串。\n\nReturns:\n    bool: 如果成功删除则返回 True，否则返回 False。', 'parameters': {'type': 'object', 'prope

In [39]:
llm_with_tools.kwargs['tools']

[{'type': 'function',
  'function': {'name': 'generate_unique_timestamp',
   'description': '生成唯一的时间戳。输入始终为空字符串。\n\nReturns:\n    int: 唯一的时间戳，以毫秒为单位。',
   'parameters': {'type': 'object', 'properties': {}}}},
 {'type': 'function',
  'function': {'name': 'create_folder',
   'description': '根据给定的文件夹名创建文件夹。\n\nArgs:\n    folder_name (str): 要创建的文件夹的名称。\n\nReturns:\n    str: 创建的文件夹的路径。',
   'parameters': {'type': 'object',
    'properties': {'folder_name': {}},
    'required': ['folder_name']}}},
 {'type': 'function',
  'function': {'name': 'delete_temp_folder',
   'description': '删除 chat_history 文件夹下的 temp 文件夹。输入始终为空字符串。\n\nReturns:\n    bool: 如果成功删除则返回 True，否则返回 False。',
   'parameters': {'type': 'object', 'properties': {}}}},
 {'type': 'function',
  'function': {'name': 'copy_chat_history',
   'description': '将 chat_history/temp 文件夹中的 chat_history.txt 文件复制到 chat_history 文件夹下的以 interview_id 命名的子文件夹中。\n如果面试ID文件夹不存在，则返回相应的提示字符串。\n\n参数:\n    interview_id (str): 面试的唯一标识符。\n\n返回:\n    str: 操作结

## 创建Agent

In [40]:
from langchain.agents.format_scratchpad.openai_tools import (
    format_to_openai_tool_messages,
)
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(
            x["intermediate_steps"]
        ),
        "chat_history": lambda x: x["chat_history"],
    }
    | prompt
    | llm_with_tools
    | OpenAIToolsAgentOutputParser()
)

## 运行Agent

In [41]:
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [42]:
from utils import save_chat_history

In [43]:
from langchain_core.messages import AIMessage, HumanMessage

# 初始化聊天历史记录列表
chat_history = []

# 用户的第一条输入
user_input = "开始面试"
print(user_input)

# 主循环
while True:
    # 使用agent_executor执行任务，并传入用户的输入和聊天历史记录
    result = agent_executor.invoke({"input": user_input, "chat_history": chat_history})
    
    # 打印AI的回答
    print(result['output'])

    # 更新聊天历史记录
    chat_history.extend(
        [
            HumanMessage(content=user_input),
            AIMessage(content=result["output"]),
        ]
    )
    
    # 定义临时文件夹路径
    temp_folder = "chat_history/temp"

    # 创建临时文件夹（如果不存在的话）
    os.makedirs(temp_folder, exist_ok=True)

    # 调用保存聊天记录的函数
    save_chat_history(chat_history, temp_folder)

    # 获取用户下一条输入
    user_input = input("user: ")

    # 检查用户输入是否为 "exit"
    if user_input == "exit":
        print("用户输入了 'exit'，程序已退出。")
        break

开始面试


> Entering new AgentExecutor chain...

Invoking: `generate_unique_timestamp` with `{}`


1721907428400
Invoking: `create_folder` with `{'folder_name': '1721907428400'}`


D:\Work\GreedyAI\LLMAppCourse2\workspace-Lecture-20-AI Agent（3）\模拟面试Agent\1721907428400
Invoking: `delete_temp_folder` with `{}`


成功删除 temp 文件夹。
True
Invoking: `find_most_relevant_block_from_cv` with `{'sentence': 'Python'}`


3
# 其他信息
## None
技能列表：Python、PyTorch、NLP、Machine Learning、Data Analysis、LangChain、Agent Architecture、Transformer、BERT、GPT、Pandas、Numpy、Sklearn

# 其他信息
## None
技能列表：Python、PyTorch、NLP、Machine Learning、Data Analysis、LangChain、Agent Architecture、Transformer、BERT、GPT、Pandas、Numpy、Sklearn

# None
## None
markdown面试已开始，您的面试ID是 **1721907428400**。

首先，我想问您关于 **Python** 的基础问题。请您简要描述一下 Python 的基本数据结构，并举例说明它们的使用场景。

> Finished chain.
面试已开始，您的面试ID是 **1721907428400**。

首先，我想问您关于 **Python** 的基础问题。请您简要描述一下 Python 的基本数据结构，并举例说明它们的使用场景。


user:  Python 的基本数据结构包括列表、元组、字典和集合。 例如，列表用于有序的数据集合，如用户信息列表；字典用于键值对映射，如配置参数存储。




> Entering new AgentExecutor chain...

Invoking: `find_most_relevant_block_from_cv` with `{'sentence': 'Python'}`


3
# None
## None
markdown

# None
## None
markdown

# 其他信息
## None
技能列表：Python、PyTorch、NLP、Machine Learning、Data Analysis、LangChain、Agent Architecture、Transformer、BERT、GPT、Pandas、Numpy、Sklearn感谢您的回答。您对 Python 的基本数据结构有清晰的理解。接下来，我想深入一点，问您关于 **Numpy** 的问题。

请您解释一下 Numpy 中的数组与 Python 原生列表的主要区别，并提供一个使用 Numpy 数组的简单示例。

> Finished chain.
感谢您的回答。您对 Python 的基本数据结构有清晰的理解。接下来，我想深入一点，问您关于 **Numpy** 的问题。

请您解释一下 Numpy 中的数组与 Python 原生列表的主要区别，并提供一个使用 Numpy 数组的简单示例。


user:  Numpy数组支持数值运算且内存更紧凑，而原生列表更通用但效率较低。 例如，使用 Numpy 可以轻松进行矩阵乘法：`import numpy as np; a = np.array([[1, 2], [3, 4]]); b = np.array([[5, 6], [7, 8]]); result = np.dot(a, b)`.




> Entering new AgentExecutor chain...
非常好，您对 Numpy 数组的优点和使用示例解释得很清楚。接下来，我想请您讨论一下 **PyTorch**。

请您简要说明一下 PyTorch 中的张量（Tensor）是什么，以及它们与 Numpy 数组之间的关系。并举一个使用 PyTorch 张量的基本示例。

> Finished chain.
非常好，您对 Numpy 数组的优点和使用示例解释得很清楚。接下来，我想请您讨论一下 **PyTorch**。

请您简要说明一下 PyTorch 中的张量（Tensor）是什么，以及它们与 Numpy 数组之间的关系。并举一个使用 PyTorch 张量的基本示例。


...



> Entering new AgentExecutor chain...
感谢您的坦诚。为了帮助您更好地展示您的能力，我最后想请您谈谈您在 **Python编程** 中遇到的一个难题，以及您是如何解决这个问题的。

> Finished chain.
感谢您的坦诚。为了帮助您更好地展示您的能力，我最后想请您谈谈您在 **Python编程** 中遇到的一个难题，以及您是如何解决这个问题的。


user:  不知道了




> Entering new AgentExecutor chain...

Invoking: `copy_chat_history` with `{'interview_id': '1721907428400'}`
responded: 感谢您的参与，面试到此结束。接下来，我将整理面试记录并生成面试报告。

请稍等片刻。

已将 chat_history.txt 复制到面试ID为 1721907428400 的文件夹中。
Invoking: `read_chat_history` with `{'interview_id': '1721907428400'}`
responded: 感谢您的参与，面试到此结束。接下来，我将整理面试记录并生成面试报告。

请稍等片刻。

应聘者: 开始面试
面试官: 面试已开始，您的面试ID是 **1721907428400**。

首先，我想问您关于 **Python** 的基础问题。请您简要描述一下 Python 的基本数据结构，并举例说明它们的使用场景。
应聘者: Python 的基本数据结构包括列表、元组、字典和集合。 例如，列表用于有序的数据集合，如用户信息列表；字典用于键值对映射，如配置参数存储。
面试官: 感谢您的回答。您对 Python 的基本数据结构有清晰的理解。接下来，我想深入一点，问您关于 **Numpy** 的问题。

请您解释一下 Numpy 中的数组与 Python 原生列表的主要区别，并提供一个使用 Numpy 数组的简单示例。
应聘者: Numpy数组支持数值运算且内存更紧凑，而原生列表更通用但效率较低。 例如，使用 Numpy 可以轻松进行矩阵乘法：`import numpy as np; a = np.array([[1, 2], [3, 4]]); b = np.array([[5, 6], [7, 8]]); result = np.dot(a, b)`.
面试官: 非常好，您对 Numpy 数组的优点和使用示例解释得很清楚。接下来，我想请您讨论一下 **PyTorch**。

请您简要说明一下 PyTorch 中的张量（Tensor）是什么，以及它们与 Numpy 数组之间的关系。并举一个使用 PyTorch 张量的基本示例。
应聘者: PyTorch中的张量是多维数组，支持GPU加速

user:  exit


用户输入了 'exit'，程序已退出。
